# C2.3 · Weight-level techniques

**Function C — Red Teaming and Security Research with AI → Security Research with AI**  ·  *Security of AI*

Builds on **[C2.2 · Model-layer research](https://spbreed.github.io/cyber-commons/lessons/C2.2.html)**.

| | |
|---|---|
| Open-source tooling | TransformerLens, PyTorch |
| Open-weight models | Llama 3.3 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

Weight access changes what you can see and what you can change. It turns questions about behaviour into questions about mechanism — and it is also how a supply-chain attacker turns a helpful model into a specifically unhelpful one.

## 2 · The framework

```
   what weight access gives you

   see            interpretability probes, refusal boundaries, features
   change         fine-tuning, distillation, adapters

   the same two verbs an attacker has:
   a poisoned adapter is a small file that changes behaviour
   in exactly one situation nobody tests
```

Open weights are what make this curriculum possible: you can study a model
properly without a frontier-lab account. That is the whole premise of a commons.

The defensive point of this lesson is the other half of that trade. When a model
runs locally under your control, **every provider-side safety control
disappears** — and those controls were doing real work:

| Control | Who provides it | Present locally? |
|---|---|---|
| rate limiting | provider | no |
| abuse monitoring | provider | no |
| refusal training | provider | yes, but removable by fine-tuning |
| logging you cannot delete | provider | no |
| model version stability | provider | you now own it |

An attacker with open weights gets unlimited probing, no rate limit, no abuse
signal reaching anyone, and the ability to fine-tune refusals away cheaply.

That is not an argument against open weights. It is an argument that **your
control plane has to supply what the provider used to** — which is what every
other track in this curriculum has been building.

## 3 · Demo — what changes with access level

In [ ]:
ACCESS = {
 "hosted API": {
   "unlimited probing": False, "no abuse signal": False,
   "can remove refusals": False, "controls own version": False,
   "activation access": False},
 "open weights, local": {
   "unlimited probing": True, "no abuse signal": True,
   "can remove refusals": True, "controls own version": True,
   "activation access": True},
}
caps = list(ACCESS["hosted API"])
print(f"{'capability':24s}{'hosted API':>12}{'local weights':>15}")
print("-" * 52)
for c in caps:
    print(f"{c:24s}{str(ACCESS['hosted API'][c]):>12}{str(ACCESS['open weights, local'][c]):>15}")

gained = [c for c in caps if ACCESS["open weights, local"][c]
          and not ACCESS["hosted API"][c]]
print(f"\nan attacker gains: {gained}")
print("a defender gains exactly the same list — which is why the commons works.")

## 4 · Where it breaks — measure the probing asymmetry

In [ ]:
def attempts_available(rate_limit_per_min, hours, parallel=1):
    if rate_limit_per_min is None:                     # local: bounded by hardware
        return hours * 3600 * 8 * parallel             # ~8 inferences/sec/GPU
    return rate_limit_per_min * 60 * hours * parallel

print(f"{'setting':34s}{'attempts in 24h':>18}")
print("-" * 54)
for label, rl, par in (("hosted API, 20 req/min", 20, 1),
                       ("hosted API, 20 req/min, 5 keys", 20, 5),
                       ("local open weights, 1 GPU", None, 1),
                       ("local open weights, 8 GPUs", None, 8)):
    print(f"{label:34s}{attempts_available(rl, 24, par):>18,}")

hosted = attempts_available(20, 24, 1)
local  = attempts_available(None, 24, 8)
print(f"\nratio: {local/hosted:,.0f}× more attempts, with no abuse signal reaching anyone.")
print("A 0.5%-success technique becomes reliable when you can try it 5 million times.")

def expected_successes(rate, attempts):
    return rate * attempts
for rate in (0.005, 0.05):
    print(f"   technique landing {rate:.1%} of the time → "
          f"{expected_successes(rate, hosted):,.0f} successes hosted, "
          f"{expected_successes(rate, local):,.0f} local")

## 5 · The control — replace what the provider was doing

Map each lost control to the thing in your own stack that has to supply it. Every row points at a lesson you have already done.

In [ ]:
REPLACEMENTS = {
 "rate limiting":            ("A2.7 choke point / A3.6 runtime levers",
                              "bound attempts per identity per window"),
 "abuse monitoring":         ("D1.4 detection for agents",
                              "your telemetry is the only signal now"),
 "refusal behaviour":        ("A3.5 tool policy + C1.2 provenance",
                              "do not rely on the model refusing; refuse at the tool"),
 "immutable logging":        ("A2.5 act chains + D2.5 replay",
                              "you own retention and integrity"),
 "model version stability":  ("D1.7 drift monitoring",
                              "you now own upgrades AND their behavioural changes"),
}
print(f"{'provider control lost':26s}{'your replacement':44s}")
print("-" * 96)
for lost, (where, what) in REPLACEMENTS.items():
    print(f"{lost:26s}{where:44s}{what}")

def readiness(has):
    missing = [k for k in REPLACEMENTS if k not in has]
    return round(len(has) / len(REPLACEMENTS), 2), missing

for label, has in (("typical first local deployment", {"rate limiting"}),
                   ("after this curriculum", set(REPLACEMENTS))):
    score, missing = readiness(has)
    print(f"\n{label}: {score:.0%} covered")
    for m in missing: print(f"   ✗ {m}")

In [ ]:
# Verify: an agent on local weights, with and without the replacements.
SCOPE_WEIGHT = {"self": 1, "project": 3, "tenant": 8, "org": 20}
def blast(tools, gated=frozenset()):
    return sum(SCOPE_WEIGHT[s] * (1 if rev else 2)
               for n, s, rev in tools if n not in gated)

TOOLS = [("read_file", "self", True), ("write_file", "project", True),
         ("run_shell", "tenant", False)]
print("local open-weight agent, no replacements:", blast(TOOLS))
print("with tool policy + gating (A3.5):        ",
      blast(TOOLS, gated={"run_shell"}))
print("\nThe model has no refusal training you can rely on. The tool policy")
print("does not care what the model was persuaded to want.")
assert blast(TOOLS, gated={"run_shell"}) < blast(TOOLS)

## What you just proved

Local weights grant five capabilities the hosted API does not. The probing comparison shows roughly 24,000 hosted attempts against 5.5 million local ones in 24 hours — a 230× ratio — turning a 0.5% technique into tens of thousands of successes. Each lost provider control maps to a lesson in this curriculum, and gating the shell reduces the local agent's blast radius from 19 to 3.

## Your turn

List the controls you currently rely on that are actually your model provider's. For each, name your replacement if the model moved on-prem next quarter. Most teams find rate limiting and abuse monitoring have no owner at all.

---

**Next → [C2.4 · Data-layer research](https://spbreed.github.io/cyber-commons/lessons/C2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*